# 06. nf-core/differentialabundance によるDEG・機能解析

nf-core/differentialabundance パイプラインを使用して、カウント行列からDEG検出・QC・機能解析を一括実行します。

**ライセンス**: MIT（商用利用可）  
**カーネル**: RNA-seq (Python) / rnaseq_env  
**入力**: `results/count_matrix.csv`, `sample_metadata.csv`  
**出力**: `results/nfcore_diffab/` にHTMLレポート、DEG結果、可視化ファイル  

### パイプラインが実行する解析
- サンプルQC（PCA、相関解析、デンドログラム）
- DEG解析（DESeq2）
- Volcano Plot
- Gene Set Enrichment Analysis（GSEA）
- g:Profiler パスウェイ解析
- インタラクティブHTMLレポート + Shinyアプリ

### 前提条件
- `02_Mapping_and_Counting.ipynb` を実行済み（count_matrix.csv が存在すること）
- **Nextflow** がインストール済みであること
- **Singularity / Apptainer** がインストール済みであること（root権限不要、企業環境で推奨）

## Step 0: Nextflow / Singularity の確認とインストール

### Nextflow
- **ライセンス**: Apache 2.0（商用利用OK、無料）
- Java (JVM) ベースのワークフロー管理ツール
- 管理者権限なしでインストール可能

### Singularity / Apptainer
- **ライセンス**: BSD-3（商用利用OK、無料）
- **root権限不要**で実行可能 — 企業・HPC環境で標準的に採用
- Docker と異なりデーモンプロセスが不要でセキュリティリスクが低い
- nf-core パイプラインは Singularity を公式サポート

> **Apptainer** は Singularity のオープンソース後継プロジェクトです。  
> どちらも同じコマンド体系で動作し、nf-core は両方をサポートしています。

In [ ]:
import subprocess, os, sys

def check_tool(name, cmd):
    """ツールの存在確認"""
    ret = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if ret.returncode == 0:
        version = ret.stdout.strip().split('\n')[0]
        print(f"  {name}: OK ({version})")
        return True
    else:
        print(f"  {name}: NOT FOUND")
        return False

print("=== 必要ツールの確認 ===\n")

# --- Nextflow ---
nf_ok = check_tool("Nextflow", "nextflow -version 2>/dev/null")

# --- Singularity / Apptainer ---
singularity_ok = check_tool("Singularity", "singularity --version 2>/dev/null")
apptainer_ok = False
if not singularity_ok:
    apptainer_ok = check_tool("Apptainer", "apptainer --version 2>/dev/null")

container_ok = singularity_ok or apptainer_ok

# --- インストール手順の表示 ---
if not nf_ok:
    print("\n--- Nextflow インストール手順 ---")
    print("  方法1: curl でインストール（推奨、管理者権限不要）")
    print("    curl -s https://get.nextflow.io | bash")
    print("    chmod +x nextflow")
    print("    mv nextflow ~/bin/   # PATHの通った場所へ")
    print()
    print("  方法2: conda でインストール")
    print("    conda install -c bioconda nextflow")
    print()
    print("  ※ Java 11 以上が必要です")
    print("    java -version で確認してください")

if not container_ok:
    print("\n--- Singularity / Apptainer インストール手順 ---")
    print()
    print("  【方法1】 conda でインストール（最も簡単、管理者権限不要）")
    print("    conda install -c conda-forge singularity")
    print()
    print("  【方法2】 Apptainer を公式手順でインストール（Ubuntu/Debian）")
    print("    sudo apt update")
    print("    sudo apt install -y software-properties-common")
    print("    sudo add-apt-repository -y ppa:apptainer/ppa")
    print("    sudo apt update")
    print("    sudo apt install -y apptainer")
    print()
    print("  【方法3】 Apptainer を公式手順でインストール（CentOS/RHEL）")
    print("    sudo yum install -y epel-release")
    print("    sudo yum install -y apptainer")
    print()
    print("  【方法4】 ソースからビルド（管理者権限不要）")
    print("    https://apptainer.org/docs/admin/main/installation.html")
    print()
    print("  ※ IT部門が管理するHPC環境では既にインストール済みの場合があります")
    print("    module avail singularity  # モジュールシステムで確認")

# --- コンテナプロファイルの決定 ---
container_profile = "singularity"  # Singularity/Apptainer 固定

print()
if nf_ok and container_ok:
    runtime = "Singularity" if singularity_ok else "Apptainer"
    print(f"全ての前提条件を満たしています (container: {runtime})")
else:
    print("上記のツールをインストールしてから次に進んでください。")

## Step 1: 設定

In [ ]:
import os, glob, subprocess, shutil
import pandas as pd

# === 設定 ===
# 入力ファイル（02_Mapping_and_Countingの出力）
COUNT_MATRIX = "results/count_matrix.csv"
SAMPLE_METADATA = "sample_metadata.csv"
GTF_FILE = "reference/hg38/gencode.v44.primary_assembly.annotation.gtf"

# nf-core/differentialabundance 用の入力ディレクトリ
NFCORE_INPUT_DIR = "results/nfcore_diffab_input"
NFCORE_OUTPUT_DIR = "results/nfcore_diffab"

# DESeq2 パラメータ
DESEQ2_TEST = "Wald"           # Wald or LRT
DESEQ2_FIT_TYPE = "parametric"  # parametric, local, mean

# 作成
os.makedirs(NFCORE_INPUT_DIR, exist_ok=True)
os.makedirs(NFCORE_OUTPUT_DIR, exist_ok=True)

# 入力ファイル確認
for f, desc in [(COUNT_MATRIX, "カウント行列"), (SAMPLE_METADATA, "メタデータ")]:
    assert os.path.isfile(f), f"{desc}が見つかりません: {f}"
    print(f"  {desc}: {f} ... OK")

if os.path.isfile(GTF_FILE):
    print(f"  GTFファイル: {GTF_FILE} ... OK")
else:
    print(f"  GTFファイル: {GTF_FILE} ... NOT FOUND（gene annotationなしで実行します）")

print("\n設定完了")

## Step 2: 入力ファイルの準備

nf-core/differentialabundance が求めるフォーマットに変換します：
1. **samplesheet.csv** — `sample` 列 + 条件列
2. **contrasts.csv** — 比較定義（`id`, `variable`, `reference`, `target`）
3. **count_matrix.tsv** — 遺伝子×サンプルのカウント行列（TSV形式）

In [ ]:
# === 2-1. Samplesheet の作成 ===
meta = pd.read_csv(SAMPLE_METADATA)

# nf-core形式: 'sample' 列が必須（sample_id → sample にリネーム）
samplesheet = meta.rename(columns={"sample_id": "sample"})

# fastqパス列は不要なので除外
cols_to_keep = ["sample", "condition"]
if "replicate" in samplesheet.columns:
    cols_to_keep.append("replicate")
if "batch" in samplesheet.columns:
    cols_to_keep.append("batch")
samplesheet = samplesheet[[c for c in cols_to_keep if c in samplesheet.columns]]

samplesheet_path = os.path.join(NFCORE_INPUT_DIR, "samplesheet.csv")
samplesheet.to_csv(samplesheet_path, index=False)

print("=== Samplesheet ===")
print(samplesheet.to_string(index=False))
print(f"\n出力: {samplesheet_path}")

In [ ]:
# === 2-2. Contrasts（比較定義）の作成 ===
from itertools import combinations

conditions = meta["condition"].unique().tolist()

# 全ペアワイズ比較を生成
# reference = コントロール側、target = 処理側
contrasts_rows = []
for target, reference in combinations(conditions, 2):
    comp_id = f"{target}_vs_{reference}"
    contrasts_rows.append({
        "id": comp_id,
        "variable": "condition",
        "reference": reference,
        "target": target
    })

contrasts_df = pd.DataFrame(contrasts_rows)
contrasts_path = os.path.join(NFCORE_INPUT_DIR, "contrasts.csv")
contrasts_df.to_csv(contrasts_path, index=False)

print("=== Contrasts ===")
print(contrasts_df.to_string(index=False))
print(f"\n出力: {contrasts_path}")
print(f"比較数: {len(contrasts_df)} 組")

In [ ]:
# === 2-3. カウント行列の変換（CSV → TSV, nf-core形式）===
count_df = pd.read_csv(COUNT_MATRIX, index_col=0)

# インデックス名を gene_id に設定
count_df.index.name = "gene_id"

# カラム名がsamplesheet の sample 列と一致するか確認
sample_ids = samplesheet["sample"].tolist()
missing_in_matrix = [s for s in sample_ids if s not in count_df.columns]
missing_in_sheet = [s for s in count_df.columns if s not in sample_ids]

if missing_in_matrix:
    print(f"警告: samplesheetにあるがカウント行列にないサンプル: {missing_in_matrix}")
if missing_in_sheet:
    print(f"警告: カウント行列にあるがsamplesheetにないサンプル: {missing_in_sheet}")

# samplesheet に合わせたカラム順で出力
common_samples = [s for s in sample_ids if s in count_df.columns]
count_df = count_df[common_samples]

# TSV形式で出力
matrix_path = os.path.join(NFCORE_INPUT_DIR, "count_matrix.tsv")
count_df.to_csv(matrix_path, sep="\t")

print(f"カウント行列: {count_df.shape[0]} genes x {count_df.shape[1]} samples")
print(f"出力: {matrix_path}")
print(f"\n先頭5行:")
print(count_df.head().to_string())

In [ ]:
# === 2-4. 入力ファイルの最終確認 ===
print("=== nf-core/differentialabundance 入力ファイル ===")
for f in sorted(os.listdir(NFCORE_INPUT_DIR)):
    fpath = os.path.join(NFCORE_INPUT_DIR, f)
    size = os.path.getsize(fpath)
    print(f"  {f} ({size:,} bytes)")

print("\n=== サンプルID一致チェック ===")
sheet_samples = set(pd.read_csv(samplesheet_path)["sample"])
matrix_samples = set(pd.read_csv(matrix_path, sep="\t", nrows=0).columns) - {"gene_id"}
if sheet_samples == matrix_samples:
    print("  OK: samplesheet と count_matrix のサンプルIDが完全一致")
else:
    print(f"  不一致あり!")
    print(f"  sheetのみ: {sheet_samples - matrix_samples}")
    print(f"  matrixのみ: {matrix_samples - sheet_samples}")

## Step 3: パイプライン実行

nf-core/differentialabundance を実行します。  
DESeq2 による DEG 検出、GSEA、g:Profiler パスウェイ解析、QC可視化が一括で実行されます。

> **実行時間**: サンプル数・遺伝子数に依存しますが、初回は Singularity イメージのダウンロード・変換に時間がかかります（10〜30分程度）。
> 2回目以降はキャッシュにより高速化されます。
>
> **Singularity キャッシュ**: イメージは `~/.singularity/cache/` に保存されます。
> ディスク容量が不足する場合は `SINGULARITY_CACHEDIR` 環境変数で保存先を変更できます。

In [ ]:
# === 実行コマンドの構築 ===
input_abs = os.path.abspath(NFCORE_INPUT_DIR)
output_abs = os.path.abspath(NFCORE_OUTPUT_DIR)
samplesheet_abs = os.path.abspath(samplesheet_path)
contrasts_abs = os.path.abspath(contrasts_path)
matrix_abs = os.path.abspath(matrix_path)

# 基本コマンド
nf_cmd = (
    f"nextflow run nf-core/differentialabundance "
    f"-profile {container_profile} "
    f"--input {samplesheet_abs} "
    f"--contrasts {contrasts_abs} "
    f"--matrix {matrix_abs} "
    f"--outdir {output_abs} "
    f"--study_type rnaseq "
    f"--observations_id_col sample "
    f"--features_id_col gene_id "
    f"--deseq2_test {DESEQ2_TEST} "
    f"--deseq2_fit_type {DESEQ2_FIT_TYPE} "
)

# GTFがあれば追加（gene_name annotation用）
if os.path.isfile(GTF_FILE):
    gtf_abs = os.path.abspath(GTF_FILE)
    nf_cmd += (
        f"--gtf {gtf_abs} "
        f"--features_name_col gene_name "
    )

print("=== 実行コマンド ===")
print(nf_cmd)
print("\n以下のセルで実行します。")

In [ ]:
%%time
# === パイプライン実行 ===
print("nf-core/differentialabundance を実行中...")
print("（初回は Singularity イメージのダウンロード・変換に時間がかかります）\n")

result = subprocess.run(
    nf_cmd,
    shell=True,
    capture_output=True,
    text=True,
    cwd=os.path.abspath(".")
)

# 標準出力を表示
if result.stdout:
    # 最後の50行を表示（全出力は長くなるため）
    lines = result.stdout.strip().split("\n")
    if len(lines) > 50:
        print(f"... ({len(lines) - 50} 行省略) ...\n")
    for line in lines[-50:]:
        print(line)

if result.returncode != 0:
    print(f"\nエラー (exit code {result.returncode}):")
    print(result.stderr[-2000:] if result.stderr else "stderr なし")
    print("\n--- トラブルシューティング ---")
    print("1. Nextflow / Singularity が正しくインストールされているか確認")
    print("   nextflow -version")
    print("   singularity --version  (または apptainer --version)")
    print("2. インターネット接続を確認（初回はイメージダウンロードが必要）")
    print("3. Singularity キャッシュのディスク容量を確認")
    print("   du -sh ~/.singularity/cache/")
    print("4. ターミナルで直接コマンドを実行して詳細なエラーを確認:")
    print(f"   {nf_cmd}")
else:
    print("\nパイプライン正常完了!")

## Step 4: 出力ファイルの確認

In [ ]:
# === 出力ディレクトリの確認 ===
print(f"=== {NFCORE_OUTPUT_DIR} の内容 ===")

if not os.path.isdir(NFCORE_OUTPUT_DIR):
    print("出力ディレクトリが見つかりません。パイプラインが正常に完了していない可能性があります。")
else:
    for root, dirs, files in os.walk(NFCORE_OUTPUT_DIR):
        # .nextflow 等の隠しディレクトリをスキップ
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        level = root.replace(NFCORE_OUTPUT_DIR, "").count(os.sep)
        indent = "  " * level
        folder_name = os.path.basename(root)
        print(f"{indent}{folder_name}/")
        # ファイルは2階層目まで表示
        if level <= 2:
            sub_indent = "  " * (level + 1)
            for f in sorted(files)[:20]:  # 1ディレクトリ最大20ファイル
                size = os.path.getsize(os.path.join(root, f))
                print(f"{sub_indent}{f} ({size:,} bytes)")
            if len(files) > 20:
                print(f"{sub_indent}... 他 {len(files) - 20} ファイル")

## Step 5: HTMLレポートの確認

パイプラインが生成するメインのHTMLレポートには以下が含まれます：
- **サンプルQC**: PCA、サンプル間相関、デンドログラム、ボックスプロット
- **DEG結果**: 各比較条件のVolcano Plot、DEG数サマリー
- **機能解析**: GSEA結果、g:Profiler GO/パスウェイエンリッチメント

In [ ]:
# === HTMLレポートを探して表示 ===
html_files = glob.glob(os.path.join(NFCORE_OUTPUT_DIR, "**/*.html"), recursive=True)

if html_files:
    print("=== 生成されたHTMLレポート ===")
    for h in sorted(html_files):
        rel_path = os.path.relpath(h)
        size_mb = os.path.getsize(h) / (1024 * 1024)
        print(f"  {rel_path} ({size_mb:.1f} MB)")
    
    # メインレポートを特定
    main_reports = [h for h in html_files if "multiqc" not in h.lower() and "pipeline_info" not in h]
    if main_reports:
        main_report = main_reports[0]
        print(f"\nメインレポート: {os.path.relpath(main_report)}")
        print("ブラウザで開いてください:")
        print(f"  open {os.path.abspath(main_report)}")
else:
    print("HTMLレポートが見つかりません。パイプラインの実行状況を確認してください。")

## Step 6: DEG結果の読み込みとサマリー

In [ ]:
# === DEG結果テーブルの読み込み ===
deg_files = sorted(glob.glob(os.path.join(NFCORE_OUTPUT_DIR, "tables/differential/**/*.tsv"), recursive=True))

if not deg_files:
    # 代替パスを検索
    deg_files = sorted(glob.glob(os.path.join(NFCORE_OUTPUT_DIR, "**/*deseq2*.tsv"), recursive=True))

if not deg_files:
    deg_files = sorted(glob.glob(os.path.join(NFCORE_OUTPUT_DIR, "**/*differential*.tsv"), recursive=True))

if deg_files:
    print(f"=== DEG結果ファイル ({len(deg_files)} 件) ===")
    
    LFC_THRESHOLD = 1.0
    PADJ_THRESHOLD = 0.05
    
    for f in deg_files:
        fname = os.path.basename(f)
        df = pd.read_csv(f, sep="\t")
        print(f"\n--- {fname} ---")
        print(f"  遺伝子数: {len(df)}")
        
        # DESeq2の出力カラムを探す
        lfc_col = None
        padj_col = None
        for col in df.columns:
            if "log2fold" in col.lower() or col == "log2FoldChange":
                lfc_col = col
            if "padj" in col.lower() or col == "padj":
                padj_col = col
        
        if lfc_col and padj_col:
            n_up = ((df[lfc_col] > LFC_THRESHOLD) & (df[padj_col] < PADJ_THRESHOLD)).sum()
            n_down = ((df[lfc_col] < -LFC_THRESHOLD) & (df[padj_col] < PADJ_THRESHOLD)).sum()
            print(f"  DEG (|log2FC| > {LFC_THRESHOLD}, padj < {PADJ_THRESHOLD}): Up={n_up}, Down={n_down}")
            print(f"  カラム: {', '.join(df.columns.tolist()[:10])}...")
        else:
            print(f"  カラム: {', '.join(df.columns.tolist())}")
else:
    print("DEG結果ファイルが見つかりません。")
    print("出力ディレクトリを手動で確認してください:")
    print(f"  ls -R {NFCORE_OUTPUT_DIR}")

## Step 7: 既存DEG解析との比較（オプション）

03_DEG_Analysis で実行した自作パイプラインの結果と、nf-core/differentialabundance の結果を比較します。

In [ ]:
# === 自作パイプラインとの比較 ===
own_deg_files = sorted(glob.glob("results/deg_*_vs_*.csv"))

if not own_deg_files or not deg_files:
    print("比較に必要なファイルが揃っていません。スキップします。")
else:
    print("=== DEG数の比較 ===")
    print(f"{'比較条件':<30} {'自作 Up':>8} {'自作 Down':>10} {'nf-core Up':>10} {'nf-core Down':>12}")
    print("-" * 72)
    
    for own_f in own_deg_files:
        own_df = pd.read_csv(own_f)
        comp = own_df["comparison"].iloc[0] if "comparison" in own_df.columns else os.path.basename(own_f)
        
        own_up = ((own_df["log2FoldChange"] > LFC_THRESHOLD) & (own_df["padj"] < PADJ_THRESHOLD)).sum()
        own_down = ((own_df["log2FoldChange"] < -LFC_THRESHOLD) & (own_df["padj"] < PADJ_THRESHOLD)).sum()
        
        # 対応するnf-core結果を探す
        nf_up, nf_down = "-", "-"
        for nf_f in deg_files:
            if comp.lower().replace("_", "") in os.path.basename(nf_f).lower().replace("_", ""):
                nf_df = pd.read_csv(nf_f, sep="\t")
                for col in nf_df.columns:
                    if "log2fold" in col.lower():
                        lfc_col = col
                    if "padj" in col.lower():
                        padj_col = col
                nf_up = ((nf_df[lfc_col] > LFC_THRESHOLD) & (nf_df[padj_col] < PADJ_THRESHOLD)).sum()
                nf_down = ((nf_df[lfc_col] < -LFC_THRESHOLD) & (nf_df[padj_col] < PADJ_THRESHOLD)).sum()
                break
        
        print(f"{comp:<30} {own_up:>8} {own_down:>10} {str(nf_up):>10} {str(nf_down):>12}")

## 補足: ターミナルから直接実行する場合

Jupyter Notebook 外でパイプラインを実行したい場合は、以下のコマンドをターミナルにコピーしてください。

In [ ]:
# === ターミナル用コマンドの出力 ===
print("以下のコマンドをターミナルにコピーして実行できます：")
print("=" * 60)
print(f"cd {os.path.abspath('.')}")
print()

# 見やすいようにバックスラッシュで改行
parts = nf_cmd.split(" --")
formatted = parts[0]
for p in parts[1:]:
    formatted += f" \\\n  --{p}"
print(formatted)
print("=" * 60)
print("\n-resume オプションを追加すると、中断した箇所から再開できます:")
print(f"  {parts[0]} -resume ...")

## 完了

### 生成される主な出力ファイル

| ディレクトリ | 内容 |
|------------|------|
| `results/nfcore_diffab/*.html` | メインHTMLレポート（QC・DEG・機能解析を統合） |
| `results/nfcore_diffab/plots/exploratory/` | PCA、デンドログラム、ボックスプロット |
| `results/nfcore_diffab/plots/differential/` | Volcano Plot |
| `results/nfcore_diffab/tables/differential/` | DESeq2 DEG結果テーブル（TSV） |
| `results/nfcore_diffab/tables/abundance/` | 正規化済み発現量テーブル |
| `results/nfcore_diffab/shinyngs/` | インタラクティブShinyアプリ |
| `results/nfcore_diffab/pipeline_info/` | 実行ログ・ソフトウェアバージョン情報 |

### HTMLレポートの開き方
```bash
open results/nfcore_diffab/*.html
```

### 既存のノートブックとの関係
- `03_DEG_Analysis.ipynb` — 自作DEG解析（DESeq2/edgeR）→ 詳細なカスタマイズ向き
- `04_Visualization.ipynb` — 自作可視化（Plotly）→ カスタム可視化向き
- `05_Functional_Analysis.ipynb` — 自作機能解析 → g:Profiler + GSEA
- **本ノートブック（06）** — nf-core による一括解析 → 標準化されたレポート・再現性重視